<a href="https://colab.research.google.com/github/Vdivyajaswanthi123/ai-mentor-portfolio/blob/main/Day5_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers requests sentence-transformers
import os, getpass
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

In [ ]:
import os
print("Token set:", bool(os.environ.get('HF_TOKEN')))

In [ ]:
import os, requests, time
from requests.exceptions import ConnectionError as ReqConnectionError

HF_TOKEN = os.environ['HF_TOKEN']  # 👈 this must run AFTER Cell 1

def hf_zero_shot_api(text, labels, retries=3):
    for attempt in range(retries):
        try:
            r = requests.post(
                'https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli',
                headers={'Authorization': f'Bearer {HF_TOKEN}'},
                json={'inputs': text, 'parameters': {'candidate_labels': labels}},
                timeout=30
            )
            result = r.json()
            if 'error' in result and 'loading' in result.get('error', '').lower():
                wait = result.get('estimated_time', 20)
                print(f"Model loading, waiting {wait}s...")
                time.sleep(wait)
            else:
                return result
        except ReqConnectionError:
            print(f"Connection failed (attempt {attempt+1}/{retries}), retrying in 5s...")
            time.sleep(5)
    return {'error': 'Failed after all retries'}

resumes = [
    'Built React dashboards for 3 startups',
    'Implemented Spring Boot microservices in Java for fintech app',
    'Trained CNN for image classification using PyTorch, 87% accuracy',
    'Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports',
    'Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x',
]
labels = ['frontend dev', 'backend dev', 'data analyst', 'ML engineer']

start = time.time()
for r in resumes:
    result = hf_zero_shot_api(r, labels)
    if isinstance(result, dict) and 'error' in result:
        print(f'  Error: {result["error"]}')
    else:
        # result is a list sorted by score
        top = result[0]                                        # 👈 list, not dict
        print(f'  {r[:50]:50} -> {top["label"]} ({top["score"]:.2f})')

In [ ]:
from transformers import pipeline

# This DOWNLOADS the model on first run — ~1.6GB. Be patient.
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

In [ ]:
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')

# Test data — 5 mock interview answers
answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')


In [ ]:
import time

def time_call(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)
    return min(times), sum(times)/len(times)

# Time API call (warm)
def call_api():
    hf_zero_shot_api('Built React dashboards', ['frontend dev', 'backend dev'])
api_min, api_avg = time_call(call_api)

# Time local call (warm)
def call_local():
    classifier('Built React dashboards', candidate_labels=['frontend dev', 'backend dev'])
local_min, local_avg = time_call(call_local)

print(f'Inference timing comparison (3 runs each, after warm-up):')
print(f'  API:   min {api_min:.2f}s | avg {api_avg:.2f}s')
print(f'  Local: min {local_min:.2f}s | avg {local_avg:.2f}s')